**Hyperparameter Tuning HandsOn**

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import loguniform

from sklearn.model_selection import(
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

**Load Data and Clean**

In [2]:
df = pd.read_csv("diabetes.csv")

In [4]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [6]:
X = df.drop(columns= ["Outcome"])
y = df["Outcome"]

**Train Test Split**

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
print("Full Dataset Size:",X.shape)
print("Training data size:",X_train.shape)
print("Test data size:",X_test.shape)

Full Dataset Size: (768, 8)
Training data size: (614, 8)
Test data size: (154, 8)


**1. Pipeline - (Scaler + Model)**

In [10]:
pipeline = Pipeline(
    [
        ("scalar",StandardScaler()),
        ("model",SVC())
    ]
)

**2. Grid Search CV(Tries all Possible Combinations)**

In [12]:
param_grid = [
    {
        "model__kernel" : ["linear"],
        "model__C" : [0.1, 1, 10, 100]
    },
    {
        "model__kernel" : ["rbf"],
        "model__C" : [0.1, 1, 10, 100],
        "model__gamma" : ["scale", "auto", 0.01, 0.1]
    },
    {
        "model__kernel" : ["poly"],
        "model__C" : [0.1, 1, 10],
        "model__gamma" : ["scale", "auto", 0.01, 0.1],
        "model__degree" : [2, 3, 4]
    }
]

In [15]:
grid = GridSearchCV(
    estimator= pipeline,
    param_grid= param_grid,
    cv = 5,
    scoring="accuracy",
    return_train_score=True,
    n_jobs= -1
)

In [16]:
grid.fit(X_train,y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...del', SVC())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'model__C': [0.1, 1, ...], 'model__kernel': ['linear']}, {'model__C': [0.1, 1, ...], 'model__gamma': ['scale', 'auto', ...], 'model__kernel': ['rbf']}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the c

In [17]:
print("GRID SEARCH CV - Results")
print("Best Params: ",grid.best_params_)
print("Best CV Accuracy: ",grid.best_score_)

GRID SEARCH CV - Results
Best Params:  {'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
Best CV Accuracy:  0.7785419165667067


In [19]:
#Evaluation on global test set(Unseen Data)
test_accuracy = grid.score(X_test, y_test)*100
print(f"Test Accuracy:{test_accuracy:.3f}")

Test Accuracy:75.325


**3. Analyzing Results**

In [20]:
grid.cv_results_

{'mean_fit_time': array([0.05064659, 0.04764218, 0.10332017, 0.58270869, 0.02645841,
        0.03007274, 0.02946386, 0.03775339, 0.04051323, 0.02810893,
        0.02715712, 0.02540207, 0.03184576, 0.03863492, 0.02516298,
        0.03337903, 0.06752925, 0.06732478, 0.03757653, 0.06944585,
        0.02597132, 0.02764654, 0.03088026, 0.02791905, 0.02722802,
        0.03100324, 0.02889037, 0.02482915, 0.03268304, 0.03104029,
        0.02549629, 0.04319611, 0.0376091 , 0.0337183 , 0.0269949 ,
        0.02924781, 0.02907057, 0.03004184, 0.04040928, 0.03518357,
        0.03645906, 0.03208179, 0.02854428, 0.03703008, 0.05486426,
        0.05694613, 0.02678456, 0.04424644, 0.05896935, 0.05705051,
        0.0245667 , 0.04080853, 0.05494418, 0.05031509, 0.02550821,
        0.04061251]),
 'std_fit_time': array([0.00956763, 0.01225033, 0.01021181, 0.16342916, 0.00455619,
        0.00750349, 0.00557097, 0.01317014, 0.00872441, 0.00392211,
        0.00300422, 0.00200679, 0.00223843, 0.00373364, 0.001

In [23]:
grid_results_df = pd.DataFrame(grid.cv_results_)

grid_results_df = grid_results_df[["params", "mean_train_score", "mean_test_score","rank_test_score"]].sort_values("rank_test_score")

grid_results_df

,params,mean_train_score,mean_test_score,rank_test_score
14,"{'model__C': 10, 'model__gamma': 0.01, 'model_...",0.800902,0.778542,1
0,"{'model__C': 0.1, 'model__kernel': 'linear'}",0.790718,0.778529,2
10,"{'model__C': 1, 'model__gamma': 0.01, 'model__...",0.789904,0.776929,3
1,"{'model__C': 1, 'model__kernel': 'linear'}",0.795197,0.775290,4
2,"{'model__C': 10, 'model__kernel': 'linear'}",0.794791,0.775290,4
3,"{'model__C': 100, 'model__kernel': 'linear'}",0.795198,0.773664,6
18,"{'model__C': 100, 'model__gamma': 0.01, 'model...",0.827775,0.762308,7
11,"{'model__C': 1, 'model__gamma': 0.1, 'model__k...",0.833473,0.758963,8
5,"{'model__C': 0.1, 'model__gamma': 'auto', 'mod...",0.779728,0.754085,9
7,"{'model__C': 0.1, 'model__gamma': 0.1, 'model_...",0.781355,0.754085,9


In [24]:
len(grid_results_df)

56

**4. Randomized Search CV(Tries Random Combinations)**

In [25]:
param_dist = [
    #Linear
    {
        "model__kernel" : ["linear"],
        "model__C" : loguniform(0.01, 100)
    },
    #RBF
    {
        "model__kernel" : ["rbf"],
        "model__C" : loguniform(0.01, 1000),
        "model__gamma" : loguniform(0.001, 10)
    },
    #Poly
    {
        "model__kernel" : ["poly"],
        "model__C" : loguniform(0.01, 100),
        "model__gamma" : loguniform(0.001, 1)
    }
]

In [26]:
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    return_train_score=True,
    n_jobs=-1
)

In [27]:
random_search.fit(X_train,y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...del', SVC())])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","[{'model__C': <scipy.stats....001FF93B15FD0>, 'model__kernel': ['linear']}, {'model__C': <scipy.stats....001FFD3B4C410>, 'model__gamma': <scipy.stats....001FFD3B4E210>, 'model__kernel': ['rbf']}, ...]"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User G

In [28]:
print("RANDOMIZED SEARCH CV - Results:")
print("Best params:", random_search.best_params_)
print("Best CV Accuracy:",random_search.best_score_)

RANDOMIZED SEARCH CV - Results:
Best params: {'model__C': np.float64(1.335916680165182), 'model__gamma': np.float64(0.00678838791242122), 'model__kernel': 'rbf'}
Best CV Accuracy: 0.7769292283086766


In [31]:
test_accuracy = round(random_search.score(X_test, y_test)*100,2)
print(f"Test Accuracy: {test_accuracy}%")

Test Accuracy: 73.38%


In [32]:
random_results_df = pd.DataFrame(random_search.cv_results_)

random_results_df = random_results_df[["params", "mean_train_score", "mean_test_score","rank_test_score"]].sort_values("rank_test_score")

random_results_df

,params,mean_train_score,mean_test_score,rank_test_score
19,"{'model__C': 1.335916680165182, 'model__gamma'...",0.789498,0.776929,1
3,"{'model__C': 2.5378155082656657, 'model__kerne...",0.794383,0.775290,2
8,"{'model__C': 1.256315277393867, 'model__kernel...",0.793569,0.775290,2
12,"{'model__C': 0.015339162591163621, 'model__ker...",0.782573,0.773677,4
6,"{'model__C': 2.950706670790534, 'model__kernel...",0.794383,0.773664,5
1,"{'model__C': 2.440060709081755, 'model__kernel...",0.793976,0.773664,5
7,"{'model__C': 4.205156450913866, 'model__gamma'...",0.835916,0.767160,7
16,"{'model__C': 26.373339933815235, 'model__gamma...",0.873377,0.745968,8
4,"{'model__C': 0.012087541473056965, 'model__gam...",0.841620,0.731347,9
10,"{'model__C': 0.6672367170464207, 'model__gamma...",0.843653,0.729721,10
